# 👾 PixelCNN from scratch

In this notebook, we'll walk through the steps required to train your own PixelCNN on the fashion MNIST dataset from scratch

The code has been adapted from the excellent [PixelCNN tutorial](https://keras.io/examples/generative/pixelcnn/) created by ADMoreau, available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import lightning as L
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

from notebooks.utils_pytorch import display

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 16
PIXEL_LEVELS = 4
N_FILTERS = 128
RESIDUAL_BLOCKS = 5
BATCH_SIZE = 128
EPOCHS = 150

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data
train_set = datasets.FashionMNIST(root="../../../data", train=True, download=True)
x_train = train_set.data.numpy()  # uint8 array of shape (60000, 28, 28)

In [ ]:
# Preprocess the data
def preprocess(imgs_int):
    # (N, 28, 28) -> (N, 1, 28, 28): PyTorch images are channels-first (NCHW)
    imgs_int = torch.tensor(imgs_int, dtype=torch.float32).unsqueeze(1)
    imgs_int = F.interpolate(
        imgs_int, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )
    imgs_int = (imgs_int / (256 / PIXEL_LEVELS)).long()  # pixel level 0..PIXEL_LEVELS-1
    imgs = imgs_int.float()
    imgs = imgs / PIXEL_LEVELS
    # The targets are class ids per pixel, without the channel axis: (N, H, W),
    # which is the shape nn.CrossEntropyLoss expects for image targets
    return imgs, imgs_int.squeeze(1)


input_data, output_data = preprocess(x_train)

In [ ]:
# Show some items of clothing from the training set
display(input_data)

## 2. Build the PixelCNN

In [ ]:
# The first layer is the PixelCNN layer. This layer simply
# builds on the 2D convolutional layer, but includes masking.
class MaskedConv2D(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # PyTorch kernels have shape (out_channels, in_channels, kH, kW)
        _, _, kh, kw = self.weight.shape
        mask = torch.zeros_like(self.weight)
        mask[:, :, : kh // 2, :] = 1.0  # all rows above the centre
        mask[:, :, kh // 2, : kw // 2] = 1.0  # pixels left of the centre
        if mask_type == "B":
            mask[:, :, kh // 2, kw // 2] = 1.0  # type B may also see the centre
        # A buffer is saved and moved to the GPU with the model, but is not trained
        self.register_buffer("mask", mask)

    def forward(self, x):
        # Multiply the kernel by the mask on every call, so masked weights stay zero
        return F.conv2d(x, self.weight * self.mask, self.bias, self.stride, self.padding)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, filters):
        super().__init__()
        self.conv1 = nn.Conv2d(filters, filters // 2, kernel_size=1)
        self.pixel_conv = MaskedConv2D(
            "B", filters // 2, filters // 2, kernel_size=3, padding="same"
        )
        self.conv2 = nn.Conv2d(filters // 2, filters, kernel_size=1)

    def forward(self, inputs):
        x = F.relu(self.conv1(inputs))
        x = F.relu(self.pixel_conv(x))
        x = F.relu(self.conv2(x))
        return inputs + x

In [ ]:
# The network is a plain nn.Module; Lightning wraps it further below.
# Unlike Keras, PyTorch layers need the number of input channels up front
pixel_cnn = nn.Sequential(
    MaskedConv2D("A", 1, N_FILTERS, kernel_size=7, padding="same"),
    nn.ReLU(),
    *[ResidualBlock(N_FILTERS) for _ in range(RESIDUAL_BLOCKS)],
    MaskedConv2D("B", N_FILTERS, N_FILTERS, kernel_size=1, stride=1, padding="valid"),
    nn.ReLU(),
    MaskedConv2D("B", N_FILTERS, N_FILTERS, kernel_size=1, stride=1, padding="valid"),
    nn.ReLU(),
    # One logit per pixel level (no softmax: nn.CrossEntropyLoss applies it)
    nn.Conv2d(N_FILTERS, PIXEL_LEVELS, kernel_size=1, stride=1, padding="valid"),
)

print(pixel_cnn)
print(f"Total params: {sum(p.numel() for p in pixel_cnn.parameters()):,}")

## 3. Train the PixelCNN <a name="train"></a>

In [ ]:
# "Compiling" in Lightning = wrapping the network in a LightningModule that
# defines the loss (training_step) and the optimizer (configure_optimizers)
class LitPixelCNN(L.LightningModule):
    def __init__(self, model, learning_rate=0.0005):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.loss_fn = nn.CrossEntropyLoss()  # = sparse categorical cross-entropy on logits

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)  # (batch, PIXEL_LEVELS, H, W)
        loss = self.loss_fn(logits, y)  # y: (batch, H, W)
        self.log("loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


model = LitPixelCNN(pixel_cnn, learning_rate=0.0005)

In [ ]:
tensorboard_logger = TensorBoardLogger(save_dir="./logs")


# Generates images pixel by pixel; Lightning calls on_train_epoch_end after each epoch
class ImageGenerator(L.Callback):
    def __init__(self, model, num_img):
        self.model = model
        self.num_img = num_img

    def sample_from(self, logits, temperature):
        # softmax(logits / T) is the same as the Keras version's probs ** (1 / T),
        # renormalised; torch.multinomial draws one sample per image in the batch
        probs = (logits / temperature).softmax(dim=-1)
        return torch.multinomial(probs, num_samples=1).squeeze(-1)

    @torch.no_grad()
    def generate(self, temperature):
        self.model.eval()
        model_device = next(self.model.parameters()).device
        generated_images = torch.zeros(
            (self.num_img, 1, IMAGE_SIZE, IMAGE_SIZE), device=model_device
        )
        batch, channels, rows, cols = generated_images.shape

        for row in range(rows):
            for col in range(cols):
                for channel in range(channels):
                    logits = self.model(generated_images)[:, :, row, col]
                    generated_images[:, channel, row, col] = self.sample_from(
                        logits, temperature
                    ).float()
                    generated_images[:, channel, row, col] /= PIXEL_LEVELS

        return generated_images.cpu()

    def on_train_epoch_end(self, trainer, pl_module):
        generated_images = self.generate(temperature=1.0)
        display(
            generated_images,
            save_to="./output/generated_img_%03d.png" % (trainer.current_epoch),
        )
        pl_module.train()  # generate() switched the model to eval mode


img_generator_callback = ImageGenerator(pixel_cnn, num_img=10)

In [ ]:
train_loader = DataLoader(
    TensorDataset(input_data, output_data), batch_size=BATCH_SIZE, shuffle=True
)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    logger=tensorboard_logger,
    callbacks=[
        img_generator_callback,
        # The plain tqdm progress bar: the Rich one (used when `rich` is installed)
        # clashes with printing/plotting from a callback inside Jupyter
        TQDMProgressBar(),
    ],
)
trainer.fit(model, train_loader)

## 4. Generate images <a name="generate"></a>

In [ ]:
generated_images = img_generator_callback.generate(temperature=1.0)

In [ ]:
display(generated_images)